In [2]:
# -*- coding: utf-8 -*-
"""
ภารกิจ: The Forge - ส่วนผสมที่ 2 (เวอร์ชันแก้ไข)
สคริปต์นี้จะทำหน้าที่เป็น "นักเขียนบท" โดยจะอ่านข้อมูลจาก CACTUS dataset
แล้วทำการจำลองการสร้าง "บทละคร (Screenplay)" ที่มี [Stage Directions] กำกับ
ตามหลักการของเปเปอร์ MIRROR (เวอร์ชันแก้ไข แก้ไขบั๊กแล้วค่ะ!)
"""

import json
from datasets import load_dataset
import random
import re

# --- การตั้งค่าเบื้องต้น ---
DATASET_NAME = "LangAGI-Lab/cactus"
OUTPUT_FILE = "cactus_screenplays_v2.jsonl"
NUM_SAMPLES_TO_PROCESS = 5 # เราจะทดลองทำแค่ 5 ตัวอย่างก่อน เพื่อความรวดเร็วนะคะ

# --- คลัง Stage Directions สำหรับการจำลอง ---
# นี่คือ "คลังความคิด" ของนักเขียนบท (LLM) ของเราค่ะ
STAGE_DIRECTIONS = {
    "negative": [
        "[Sighs heavily]", "[Looks away]", "[Voice trembles slightly]",
        "[Hesitates]", "[Frowns slightly]", "[Shakes head slowly]",
        "[Sounds defensive]", "[Speaks in a low, tired voice]"
    ],
    "neutral": [
        "[Takes a moment to think]", "[Nods slowly]", "[Speeches thoughtfully]",
        "[Pauses]", "[Looks down for a second]", "[Takes a deep breath]"
    ],
    "positive": [
        "[Nods enthusiastically]", "[Smiles faintly]", "[Sounds more hopeful]",
        "[Voice brightens]", "[Speks with more energy]", "[Laughs softly]"
    ]
}

def simulate_llm_screenplay_generation(dialogue_text, attitude):
    """
    ฟังก์ชันนี้จะจำลองการทำงานของ LLM ในการแทรก Stage Directions
    เข้าไปในบทสนทนาเดิม (เวอร์ชันแก้ไข)
    """
    lines = dialogue_text.strip().split('\n')
    new_dialogue_lines = []
    
    for line in lines:
        # ใช้ regular expression ในการแยก Client/Counselor ออกจากเนื้อหา
        match = re.match(r'^(Client|Counselor):\s*(.*)', line.strip())
        if match:
            speaker, content = match.groups()
            
            # สุ่มแทรก Stage Direction เข้าไปเฉพาะในส่วนของ Client
            # โดยมีโอกาส 50% ที่จะแทรกในแต่ละประโยค
            if speaker == "Client" and random.random() < 0.5:
                direction = random.choice(STAGE_DIRECTIONS.get(attitude, STAGE_DIRECTIONS["neutral"]))
                # สร้างบรรทัดใหม่ที่มี Stage Direction แทรกอยู่
                new_line = f"Client: {direction} {content}"
                new_dialogue_lines.append(new_line)
            else:
                # ถ้าไม่ใช่ Client หรือสุ่มไม่ติด ก็ใช้บรรทัดเดิม
                new_dialogue_lines.append(line)
        else:
            # กรณีที่บรรทัดนั้นไม่ตรงกับรูปแบบที่คาดไว้
            new_dialogue_lines.append(line)
            
    return "\n".join(new_dialogue_lines)


# --- ส่วนหลักของโปรแกรม ---
def main():
    print("🔥 เริ่มภารกิจ 'The Forge' ส่วนที่ 2: สร้างบทละคร (Screenplay Generation)... (เวอร์ชันแก้ไข!)")
    
    try:
        # 1. โหลด CACTUS dataset
        print(f"กำลังโหลด dataset: {DATASET_NAME}...")
        dataset = load_dataset(DATASET_NAME, trust_remote_code=True)
        print("โหลด Dataset สำเร็จ!")

        # 2. เตรียมไฟล์สำหรับบันทึกผลลัพธ์
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            print(f"กำลังประมวลผล {NUM_SAMPLES_TO_PROCESS} ตัวอย่างแรก และบันทึกลงใน '{OUTPUT_FILE}'...")
            
            for i in range(NUM_SAMPLES_TO_PROCESS):
                sample = dataset['train'][i]
                
                original_dialogue = sample['dialogue']
                attitude = sample['attitude']
                
                # 3. เรียกใช้ "นักเขียนบท AI" เวอร์ชันแก้ไข
                generated_screenplay = simulate_llm_screenplay_generation(original_dialogue, attitude)
                
                output_record = {
                    "original_dialogue": original_dialogue,
                    "attitude": attitude,
                    "generated_screenplay": generated_screenplay
                }
                
                f.write(json.dumps(output_record, ensure_ascii=False) + '\n')

                # แสดงตัวอย่างที่ 1 ให้ดู
                if i == 0:
                    print("\n--- ✨ ตัวอย่างผลลัพธ์ที่ 1 (เวอร์ชันแก้ไข) ✨ ---")
                    print("\n[Original Dialogue - Client's first line]")
                    print(original_dialogue.split('\n')[1])
                    
                    print("\n[Generated Screenplay - Client's first line]")
                    print(generated_screenplay.split('\n')[1])
                    print("-" * 30)

        print(f"\n✅ ภารกิจสำเร็จ! บทละครทั้ง {NUM_SAMPLES_TO_PROCESS} เรื่องถูกสร้างและบันทึกเรียบร้อยแล้วค่ะ!")
        print(f"ลองเปิดไฟล์ '{OUTPUT_FILE}' ดูได้เลยนะคะ ครั้งนี้แตกต่างกันแน่นอนค่ะ!")

    except Exception as e:
        print(f"\n❌ เกิดข้อผิดพลาดขึ้นค่ะ: {e}")

if __name__ == "__main__":
    main()


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'LangAGI-Lab/cactus' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


🔥 เริ่มภารกิจ 'The Forge' ส่วนที่ 2: สร้างบทละคร (Screenplay Generation)... (เวอร์ชันแก้ไข!)
กำลังโหลด dataset: LangAGI-Lab/cactus...
โหลด Dataset สำเร็จ!
กำลังประมวลผล 5 ตัวอย่างแรก และบันทึกลงใน 'cactus_screenplays_v2.jsonl'...

--- ✨ ตัวอย่างผลลัพธ์ที่ 1 (เวอร์ชันแก้ไข) ✨ ---

[Original Dialogue - Client's first line]
Client: Hi. I've been really anxious about going back to the animal shelter where I volunteer. I feel like the animals will hate me because they didn't remember me the last time I visited. It's been really tough.

[Generated Screenplay - Client's first line]
Client: Hi. I've been really anxious about going back to the animal shelter where I volunteer. I feel like the animals will hate me because they didn't remember me the last time I visited. It's been really tough.
------------------------------

✅ ภารกิจสำเร็จ! บทละครทั้ง 5 เรื่องถูกสร้างและบันทึกเรียบร้อยแล้วค่ะ!
ลองเปิดไฟล์ 'cactus_screenplays_v2.jsonl' ดูได้เลยนะคะ ครั้งนี้แตกต่างกันแน่นอนค่ะ!
